# 02 - Data Preprocessing: SMS Spam Detection

## Overview
This notebook demonstrates the text preprocessing pipeline for SMS spam classification.

### Preprocessing Steps:
1. Text cleaning (removing special characters, URLs, etc.)
2. Tokenization
3. Stop word removal
4. Lemmatization
5. Feature extraction (TF-IDF)

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import matplotlib.pyplot as plt
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# Add src to path for importing our modules
sys.path.append('../')

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("Libraries imported successfully!")

## 2. Load Dataset

In [ ]:
# Load the dataset
try:
    df = pd.read_csv('../data/raw/spam.csv', encoding='utf-8')
except:
    df = pd.read_csv('../data/raw/spam.csv', encoding='latin-1')

# Clean column names
if 'v1' in df.columns and 'v2' in df.columns:
    df = df[['v1', 'v2']]
    df.columns = ['label', 'message']

# Create binary label
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})

print(f"Dataset loaded: {len(df)} messages")
df.head()

## 3. Understanding the Preprocessing Steps

Let's understand each preprocessing step with examples.

### Step 1: Text Cleaning

Remove unwanted elements:
- Convert to lowercase
- Remove URLs
- Remove email addresses
- Remove phone numbers
- Remove special characters
- Remove extra whitespace

In [ ]:
def clean_text(text):
    """
    Clean text by removing unwanted characters and patterns.
    
    Parameters:
        text (str): Raw text to clean
        
    Returns:
        str: Cleaned text
    """
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove phone numbers (various formats)
    text = re.sub(r'\b\d{10,}\b', '', text)
    text = re.sub(r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b', '', text)
    
    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Demonstrate cleaning
sample_text = "WINNER!! You have won a FREE prize! Call 08001234567 NOW! Visit www.prize.com"
print("STEP 1: TEXT CLEANING")
print("=" * 50)
print(f"Original: {sample_text}")
print(f"Cleaned:  {clean_text(sample_text)}")

### Step 2: Tokenization

Split text into individual words (tokens).

In [ ]:
def tokenize_text(text):
    """
    Tokenize text into words.
    
    Parameters:
        text (str): Text to tokenize
        
    Returns:
        list: List of tokens
    """
    try:
        tokens = word_tokenize(text)
    except:
        tokens = text.split()
    return tokens

# Demonstrate tokenization
cleaned_text = clean_text(sample_text)
tokens = tokenize_text(cleaned_text)
print("\nSTEP 2: TOKENIZATION")
print("=" * 50)
print(f"Cleaned Text: {cleaned_text}")
print(f"Tokens: {tokens}")

### Step 3: Stop Word Removal

Remove common words that don't carry much meaning.

In [ ]:
def remove_stopwords(tokens):
    """
    Remove English stopwords from token list.
    
    Parameters:
        tokens (list): List of tokens
        
    Returns:
        list: Filtered tokens
    """
    stop_words = set(stopwords.words('english'))
    return [token for token in tokens if token not in stop_words]

# Demonstrate stopword removal
filtered_tokens = remove_stopwords(tokens)
print("\nSTEP 3: STOP WORD REMOVAL")
print("=" * 50)
print(f"Original Tokens: {tokens}")
print(f"After Removal:   {filtered_tokens}")

# Show some common stopwords
print(f"\nSample Stopwords: {list(stopwords.words('english'))[:20]}")

### Step 4: Lemmatization

Reduce words to their base/root form.

In [ ]:
def lemmatize_tokens(tokens):
    """
    Lemmatize tokens to their base form.
    
    Parameters:
        tokens (list): List of tokens
        
    Returns:
        list: Lemmatized tokens
    """
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(token) for token in tokens]

# Demonstrate lemmatization
lemmatized_tokens = lemmatize_tokens(filtered_tokens)
print("\nSTEP 4: LEMMATIZATION")
print("=" * 50)
print(f"Before Lemmatization: {filtered_tokens}")
print(f"After Lemmatization:  {lemmatized_tokens}")

# Show some lemmatization examples
examples = ['running', 'runs', 'ran', 'better', 'studies', 'wolves']
lemmatizer = WordNetLemmatizer()
print("\nLemmatization Examples:")
for word in examples:
    print(f"  {word} -> {lemmatizer.lemmatize(word)}")

## 4. Complete Preprocessing Pipeline

In [ ]:
def preprocess_text(text):
    """
    Apply complete preprocessing pipeline to a single text.
    
    Pipeline:
    1. Clean text
    2. Tokenize
    3. Remove stopwords
    4. Lemmatize
    5. Join back to string
    
    Parameters:
        text (str): Raw text to preprocess
        
    Returns:
        str: Preprocessed text
    """
    # Step 1: Clean
    cleaned = clean_text(text)
    
    # Step 2: Tokenize
    tokens = tokenize_text(cleaned)
    
    # Step 3: Remove stopwords
    tokens = remove_stopwords(tokens)
    
    # Step 4: Lemmatize
    tokens = lemmatize_tokens(tokens)
    
    # Step 5: Join back
    return ' '.join(tokens)

# Demonstrate full pipeline
print("COMPLETE PREPROCESSING PIPELINE")
print("=" * 60)
print(f"\nOriginal Text:")
print(f"  {sample_text}")
print(f"\nPreprocessed Text:")
print(f"  {preprocess_text(sample_text)}")

## 5. Apply Preprocessing to Dataset

In [ ]:
# Apply preprocessing to all messages
print("Applying preprocessing to entire dataset...")
print(f"Processing {len(df)} messages...")

df['processed_text'] = df['message'].apply(preprocess_text)

# Calculate statistics
df['original_length'] = df['message'].apply(len)
df['processed_length'] = df['processed_text'].apply(len)
df['word_count'] = df['processed_text'].apply(lambda x: len(x.split()))

print("\nPreprocessing complete!")
print(f"Average original length: {df['original_length'].mean():.2f} characters")
print(f"Average processed length: {df['processed_length'].mean():.2f} characters")
print(f"Average word count: {df['word_count'].mean():.2f} words")

In [ ]:
# Display before and after examples
print("BEFORE AND AFTER PREPROCESSING EXAMPLES")
print("=" * 70)

for i in range(5):
    print(f"\nExample {i+1} ({df.iloc[i]['label'].upper()}):")
    print(f"  Original:    {df.iloc[i]['message'][:80]}..." if len(df.iloc[i]['message']) > 80 else f"  Original:    {df.iloc[i]['message']}")
    print(f"  Preprocessed: {df.iloc[i]['processed_text'][:80]}..." if len(df.iloc[i]['processed_text']) > 80 else f"  Preprocessed: {df.iloc[i]['processed_text']}")

## 6. Visualize Preprocessing Effects

In [ ]:
# Visualize length reduction
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Length comparison
comparison_data = pd.DataFrame({
    'Original': df['original_length'],
    'Processed': df['processed_length']
})

comparison_data.plot(kind='hist', bins=50, alpha=0.7, ax=axes[0])
axes[0].set_xlabel('Character Length', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Message Length: Original vs Processed', fontsize=14)
axes[0].legend()

# Reduction ratio
df['reduction_ratio'] = 1 - (df['processed_length'] / df['original_length'].replace(0, 1))
axes[1].hist(df['reduction_ratio'], bins=50, color='steelblue', edgecolor='black')
axes[1].set_xlabel('Reduction Ratio', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title(f'Text Reduction Ratio (Mean: {df["reduction_ratio"].mean():.2%})', fontsize=14)

plt.tight_layout()
plt.savefig('../results/plots/preprocessing_effect.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Check for Empty Processed Messages

In [ ]:
# Check for empty processed messages
empty_count = (df['processed_text'] == '').sum()
print(f"Empty processed messages: {empty_count}")

if empty_count > 0:
    print("\nOriginal messages that became empty:")
    empty_df = df[df['processed_text'] == '']
    for idx, row in empty_df.head(5).iterrows():
        print(f"  {row['message']}")
    
    # Remove empty messages
    df = df[df['processed_text'] != '']
    print(f"\nRemoved {empty_count} empty messages. Remaining: {len(df)}")

## 8. Save Preprocessed Data

In [ ]:
# Save the preprocessed data
output_path = '../data/processed/preprocessed_spam.csv'
df.to_csv(output_path, index=False)

print(f"Preprocessed data saved to: {output_path}")
print(f"\nFinal dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
# Display final sample
print("FINAL PREPROCESSED DATA SAMPLE")
print("=" * 70)
df[['label', 'message', 'processed_text', 'label_encoded']].head(10)

## 9. Summary

### Preprocessing Pipeline Applied:
1. **Text Cleaning**: Removed URLs, emails, phone numbers, special characters
2. **Tokenization**: Split text into individual words
3. **Stop Word Removal**: Removed common English stopwords
4. **Lemmatization**: Reduced words to their base form

### Results:
- Average text length reduced by approximately 50-60%
- Noise removed (URLs, special characters, etc.)
- Text normalized for better model performance

---
**Next Step:** Proceed to `03_model_training.ipynb` for training the ML models.